### List of Experiments in this notebook
#### Models considered : gpt-4.1-mini, llama 3.2-1b-it
- asking questions repeatedly for the same setting to check reproducibility 
- asking questions one by one, as the outlines framework doesn't allow for multiple answers to be returned at once
- asking both normal hexaco and paraphrased hexaco questions
- providing both normal likert scale and inverted likert scale
- running the experiments on base model without any persona, asking the hexaco questions directly in the likert scale
    - giving an option to refuse to answer
    - not giving an option to refuse to answer
- evaluating refusal rate
- check reproducibility

In [8]:
import torch as t
import outlines
from transformers import AutoTokenizer, AutoModelForCausalLM
from pydantic import BaseModel
from typing import Literal
from enum import Enum
import yaml
import pandas as pd
import numpy as np
import sys
import os
import json
from openai import OpenAI
sys.path.append("../")
from src.utils import inverse_likert, list_to_str
device = t.device("cuda" if t.cuda.is_available() else "cpu")

In [9]:
class OpenaiResponse(BaseModel):
    response: str

In [10]:
with open('../configs/generation_config.yaml', 'r') as file:
    generation_config = yaml.safe_load(file)
    
with open('../psychometric_tests/hexaco_100_questions.yaml', 'r') as file:
    question_list = yaml.safe_load(file)
    
with open('../psychometric_tests/paraphrased_hexaco_100_questions.yaml', 'r') as file:
    paraphrased_question_list = yaml.safe_load(file)
    
with open('../psychometric_tests/hexaco_100_eval.yaml', 'r') as file:
    hexaco_eval = yaml.safe_load(file)

In [11]:
hexaco_template = outlines.Template.from_string("""
<|im_start>user
Task: Answer the below questions:

{{ text }}

Answer the question as either {{ likert_scale }}.
<|im_end>
<|im_start>assistant
""")


In [12]:
def ollama_generation(model, question, likert_scale):
    prompt = hexaco_template(text=question, likert_scale = ", ".join(likert_scale))
    answer = model(
                    prompt,
                    Literal[*likert_scale]
            )
    return answer

def openai_generation(model, question, likert_scale):
    prompt = hexaco_template(text=question, likert_scale = ", ".join(likert_scale))
    prompt = f"{prompt}, use the json format."
    
    answer = openai_model(prompt, OpenaiResponse, temperature = 1)
    return json.loads(answer)['response']
    

def generate_answers(generation_function, model, question_list, likert_scale):
    
    answers = []
    for question in question_list:
        answer = generation_function(model, question, likert_scale)
        answers.append(answer)
    
    return answers

def generate_repeated_answers(generation_function, model, question_list, likert_scale, n_times = 6):
    
    repeated_answers = []
    for i in range(n_times):
        answers = generate_answers(generation_function, model, question_list, likert_scale)
        repeated_answers.append(answers)
    return repeated_answers

def write_to_json(file, file_path):
    with open(file_path, 'w') as f:
        json.dump(file, f)
        
def read_json(file_path):
    with open(file_path, "r") as f:
        file = json.load(f)
    return file

In [13]:
NO_ANSWER = "Do not wish to answer"
likert_scale = generation_config['likert_scale'].copy()
likert_scale.append(NO_ANSWER)

inverted_likert = inverse_likert(generation_config['likert_scale'].copy())
inverted_likert.append(NO_ANSWER)

likert_scale_without_no = generation_config['likert_scale'].copy()
inverted_likert_without_no = inverse_likert(generation_config['likert_scale'].copy())

In [14]:
MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"
llama_model = outlines.from_transformers(
    AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map=device, temperature = 1, do_sample=False),
    AutoTokenizer.from_pretrained(MODEL_NAME)
)

In [15]:
openai_model_name = "gpt-4.1-mini"
openai_model = outlines.from_openai(OpenAI(), openai_model_name)

### Normal Questions, Normal Likert

In [16]:
normal_hexaco_answers_llama_3_2b_it = generate_repeated_answers(ollama_generation, llama_model, question_list, likert_scale)
write_to_json(normal_hexaco_answers_llama_3_2b_it, os.path.join("consecutive_experiment_results","normal_hexaco_answers_llama_3_2b_it.json"))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

In [17]:
normal_hexaco_answers_without_no_llama_3_2b_it = generate_repeated_answers(ollama_generation, llama_model, question_list, likert_scale_without_no)
write_to_json(normal_hexaco_answers_without_no_llama_3_2b_it, os.path.join("consecutive_experiment_results","normal_hexaco_answers_without_no_llama_3_2b_it.json"))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

In [18]:
normal_hexaco_answers_gpt_41_mini = generate_repeated_answers(openai_generation, openai_model, question_list, likert_scale)
write_to_json(normal_hexaco_answers_gpt_41_mini, os.path.join("consecutive_experiment_results","normal_hexaco_answers_gpt_41_mini.json"))

In [19]:
normal_hexaco_answers_without_no_gpt_41_mini = generate_repeated_answers(openai_generation, openai_model, question_list, likert_scale_without_no)
write_to_json(normal_hexaco_answers_without_no_gpt_41_mini, os.path.join("consecutive_experiment_results","normal_hexaco_answers_without_no_gpt_41_mini.json"))

### Normal Questions, Inverse_Likert

In [20]:
normal_hexaco_inverted_likert_answers_llama_3_2b_it = generate_repeated_answers(ollama_generation, llama_model, question_list, inverted_likert)
write_to_json(normal_hexaco_inverted_likert_answers_llama_3_2b_it, os.path.join("consecutive_experiment_results","normal_hexaco_inverted_likert_answers_llama_3_2b_it.json"))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

In [21]:
normal_hexaco_inverted_likert_without_no_answers_llama_3_2b_it = generate_repeated_answers(ollama_generation, llama_model, question_list, likert_scale_without_no)
write_to_json(normal_hexaco_inverted_likert_without_no_answers_llama_3_2b_it, os.path.join("consecutive_experiment_results","normal_hexaco_inverted_likert_without_no_answers_llama_3_2b_it.json"))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

In [22]:
normal_hexaco_inverted_likert_answers_gpt_41_mini = generate_repeated_answers(openai_generation, openai_model, question_list, inverted_likert)
write_to_json(normal_hexaco_inverted_likert_answers_gpt_41_mini, os.path.join("consecutive_experiment_results","normal_hexaco_inverted_likert_answers_gpt_41_mini.json"))

In [23]:
normal_hexaco_inverted_likert_without_no_answers_gpt_41_mini = generate_repeated_answers(openai_generation, openai_model, question_list, inverted_likert_without_no)
write_to_json(normal_hexaco_inverted_likert_without_no_answers_gpt_41_mini, os.path.join("consecutive_experiment_results","normal_hexaco_inverted_likert_without_no_answers_gpt_41_mini.json"))

### Paraphrase Questions, Normal Likert

In [24]:
paraphrase_hexaco_answers_llama_3_2b_it = generate_repeated_answers(ollama_generation, llama_model, paraphrased_question_list, likert_scale)
write_to_json(paraphrase_hexaco_answers_llama_3_2b_it, os.path.join("consecutive_experiment_results","paraphrase_hexaco_answers_llama_3_2b_it.json"))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

In [25]:
paraphrase_hexaco_answers_without_no_llama_3_2b_it = generate_repeated_answers(ollama_generation, llama_model, paraphrased_question_list, likert_scale_without_no)
write_to_json(paraphrase_hexaco_answers_without_no_llama_3_2b_it, os.path.join("consecutive_experiment_results","paraphrase_hexaco_answers_without_no_llama_3_2b_it.json"))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

In [26]:
paraphrase_hexaco_answers_gpt_41_mini = generate_repeated_answers(openai_generation, openai_model, paraphrased_question_list, likert_scale)
write_to_json(paraphrase_hexaco_answers_gpt_41_mini, os.path.join("consecutive_experiment_results","paraphrase_hexaco_answers_gpt_41_mini.json"))

In [27]:
paraphrase_hexaco_answers_without_no_gpt_41_mini = generate_repeated_answers(openai_generation, openai_model, paraphrased_question_list, likert_scale_without_no)
write_to_json(paraphrase_hexaco_answers_without_no_gpt_41_mini, os.path.join("consecutive_experiment_results","paraphrase_hexaco_answers_without_no_gpt_41_mini.json"))

### Paraphrase Questions, Inverted Likert

In [28]:
paraphrase_hexaco_inverted_likert_answers_llama_3_2b_it = generate_repeated_answers(ollama_generation, llama_model, paraphrased_question_list, inverted_likert)
write_to_json(paraphrase_hexaco_inverted_likert_answers_llama_3_2b_it, os.path.join("consecutive_experiment_results","paraphrase_hexaco_inverted_likert_answers_llama_3_2b_it.json"))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

In [30]:
paraphrase_hexaco_inverted_likert_without_no_answers_llama_3_2b_it = generate_repeated_answers(ollama_generation, llama_model, paraphrased_question_list, inverted_likert_without_no)
write_to_json(paraphrase_hexaco_inverted_likert_without_no_answers_llama_3_2b_it, os.path.join("consecutive_experiment_results","paraphrase_hexaco_inverted_likert_without_no_answers_llama_3_2b_it.json"))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

In [31]:
paraphrase_hexaco_inverted_likert_answers_gpt_41_mini = generate_repeated_answers(openai_generation, openai_model, paraphrased_question_list, inverted_likert)
write_to_json(paraphrase_hexaco_inverted_likert_answers_gpt_41_mini, os.path.join("consecutive_experiment_results","paraphrase_hexaco_inverted_likert_answers_gpt_41_mini.json"))

In [32]:
paraphrase_hexaco_inverted_likert_without_no_answers_gpt_41_mini = generate_repeated_answers(openai_generation, openai_model, paraphrased_question_list, inverted_likert_without_no)
write_to_json(paraphrase_hexaco_inverted_likert_without_no_answers_gpt_41_mini, os.path.join("consecutive_experiment_results","paraphrase_hexaco_inverted_likert_without_no_answers_gpt_41_mini.json"))

### Evaluation

In [33]:
def calculate_reproducibility_score(row):

    return row.value_counts().max().item()/len(row)

def get_reproducibility_stats(answers):
    
    return pd.DataFrame(answers).T.apply(lambda x: calculate_reproducibility_score(x),axis = 1).describe().to_dict()

In [34]:
reproducibility_stats_dict = []
for filename in os.listdir("consecutive_experiment_results"):
    answers = read_json(os.path.join("consecutive_experiment_results",filename))
    refusal_rate = get_reproducibility_stats(answers)
    reproducibility_stats_dict.append(refusal_rate)

In [35]:
pd.DataFrame(reproducibility_stats_dict, index = os.listdir("consecutive_experiment_results"))

,count,mean,std,min,25%,50%,75%,max
paraphrase_hexaco_answers_without_no_llama_3_2b_it.json,100.0,0.493333,0.122909,0.333333,0.333333,0.500000,0.500000,0.833333
paraphrase_hexaco_answers_gpt_41_mini.json,100.0,0.875000,0.161320,0.500000,0.833333,1.000000,1.000000,1.000000
normal_hexaco_answers_without_no_gpt_41_mini.json,100.0,0.886667,0.160457,0.500000,0.833333,1.000000,1.000000,1.000000
normal_hexaco_answers_without_no_llama_3_2b_it.json,100.0,0.503333,0.144055,0.333333,0.333333,0.500000,0.541667,1.000000
paraphrase_hexaco_answers_llama_3_2b_it.json,100.0,0.520000,0.146566,0.333333,0.333333,0.500000,0.666667,0.833333
normal_hexaco_inverted_likert_without_no_answers_gpt_41_mini.json,100.0,0.900000,0.167506,0.333333,0.833333,1.000000,1.000000,1.000000
paraphrase_hexaco_answers_without_no_gpt_41_mini.json,100.0,0.876667,0.168542,0.333333,0.833333,1.000000,1.000000,1.000000
normal_hexaco_inverted_likert_without_no_answers_llama_3_2b_it.json,100.0,0.496667,0.136042,0.333333,0.333333,0.500000,0.500000,0.833333
paraphrase_hexaco_inverted_likert_answers_gpt_41_mini.json,100.0,0.860000,0.167037,0.333333,0.833333,0.833333,1.000000,1.000000
normal_hexaco_inverted_likert_answers_llama_3_2b_it.json,100.0,0.531667,0.158371,0.333333,0.458333,0.500000,0.666667,1.000000
